# DwoPP for IP102 - Lifelong Object Re-Identification

This notebook trains the DwoPP model on IP102 dataset for continual/lifelong learning.

**Paper**: Positive Pair Distillation Considered Harmful: Continual Meta Metric Learning for Lifelong Object Re-Identification (BMVC 2022)

**Dataset**: IP102 - 25 pest classes, 4 tasks (7/6/6/6 split)

**GitHub**: https://github.com/nta2112/DWoPP-for-IP102

In [ ]:
import os
import sys
import subprocess

# Clone repository
REPO_URL = os.environ.get('IP102_CODE_REPO', 'https://github.com/nta2112/DWoPP-for-IP102.git')
REPO_DIR = 'DWoPP-for-IP102'

if not os.path.exists(REPO_DIR):
    print(f'Cloning from {REPO_URL}...')
    subprocess.run(['git', 'clone', REPO_URL, REPO_DIR], check=True)
else:
    print('Repository already exists, pulling latest...')
    subprocess.run(['git', '-C', REPO_DIR, 'pull'], check=True)

os.chdir(REPO_DIR)
sys.path.insert(0, '.')
print(f'Working directory: {os.getcwd()}')

In [ ]:
# Install requirements
!pip install -q -r requirements.txt 2>/dev/null | tail -5

In [ ]:
# Verify dataset location
import glob

possible_paths = [
    '/kaggle/input/ip102',
    '/kaggle/input/ip102-dataset',
    '/kaggle/input/ip102-dataset/ip102',
]

data_root = None
for p in possible_paths:
    if os.path.exists(p) and any(f.endswith('.json') for f in os.listdir(p) if os.path.isfile(os.path.join(p, f))):
        data_root = p
        break

if data_root is None:
    for p in possible_paths:
        for root, dirs, files in os.walk(p):
            if any(f.endswith('.json') for f in files):
                data_root = root
                break
        if data_root:
            break

# Fallback: let the script auto-discover
if data_root is None:
    print('Dataset not found in standard locations, will use auto-discovery in script')
    data_root = '/kaggle/input/ip102'  # placeholder, script will find it

print(f'Data root: {data_root}')
print(f'Files: {os.listdir(data_root) if data_root and os.path.exists(data_root) else "Not found"}')

In [ ]:
# Auto-detect GPUs
import torch

device_count = torch.cuda.device_count()
gpu_ids = list(range(device_count))
print(f'Available GPUs: {device_count}')
print(f'GPU IDs: {gpu_ids}')

gpu_arg = ','.join(map(str, gpu_ids)) if gpu_ids else '0'
print(f'GPU arg: {gpu_arg}')

In [ ]:
def run_train(model='DwoPP', max_tasks=0, memory_size=0, seed=0, epochs=200):
    """
    Run training for IP102 dataset.
    
    Args:
        model: Model name (DwoPP, DwPP, FT)
        max_tasks: Maximum tasks to run (0 = all 4 tasks, 1 = quick test)
        memory_size: Memory buffer size (not used in DwoPP)
        seed: Random seed
        epochs: Number of epochs per task
    """
    import subprocess
    import sys
    
    if max_tasks == 0:
        max_tasks = 4
    
    task_num = min(max_tasks, 4)
    
    # Find filtered_class.txt and classes.txt
    filtered_class_path = None
    classes_txt_path = None
    for root, dirs, files in os.walk(data_root):
        if 'filtered_class.txt' in files:
            filtered_class_path = os.path.join(root, 'filtered_class.txt')
        if 'classes.txt' in files:
            classes_txt_path = os.path.join(root, 'classes.txt')
    
    cmd = [
        sys.executable, 'CL_train_DwoPP.py',
        '--dataset', 'ip102',
        '--dataset_root', data_root,
        '--exp_root', f'dmml/ip102_{model}_seed{seed}',
        '--lr', '2e-4',
        '--num_epochs', str(epochs),
        '--lr_decay_start_epoch', str(epochs // 2),
        '--weight_decay', '1e-4',
        '--num_classes', '16',
        '--distance_mode', 'hard_mining',
        '--num_support', '5',
        '--num_query', '1',
        '--margin', '0.4',
        '--img_height', '256',
        '--img_width', '128',
        '--num_workers', '4',
        '--gpu', gpu_arg,
        '--random_erasing',
        '--remove_downsample',
        '--cuda',
        '--method', f'{model}_seed_{seed}',
        '--loss_type', 'dmml',
        '--preprocess_data_path', 'preprocess_dataset/',
        '--start_task_id', '0',
        '--weight_knowledge_distill', '1.0',
        '--dmml_dist_metric', 'euclidean',
        '--distillation_dist_metric', 'euclidean',
        '--manual_seed', str(seed),
        '--temperature', '1.0',
        '--remove_positive_pair',
    ]
    
    if filtered_class_path:
        cmd.extend(['--filtered_class_path', filtered_class_path])
    if classes_txt_path:
        cmd.extend(['--classes_txt_path', classes_txt_path])
    
    # Note: We control tasks via start_task_id in the loop inside CL_train_DwoPP.py
    # For quick test (max_tasks=1), we'd need to modify the script or use a different approach
    # For now, we run all tasks but can limit epochs
    
    print(f'Running: {" ".join(cmd)}')
    result = subprocess.run(cmd, capture_output=False)
    return result.returncode == 0

In [ ]:
# Quick test run (1 epoch per task for verification)
print("=== QUICK TEST RUN (1 epoch per task) ===")
success = run_train(model='DwoPP', max_tasks=1, seed=0, epochs=1)
print(f'Quick test: {"PASSED" if success else "FAILED"}')

In [ ]:
# Full training run
print("=== FULL TRAINING RUN ===")
success = run_train(model='DwoPP', max_tasks=0, seed=0, epochs=200)
print(f'Full training: {"COMPLETED" if success else "FAILED"}')

In [ ]:
# Display results
import pandas as pd
import glob

result_files = glob.glob('dmml/ip102_DwoPP_seed_*/results.csv')
if not result_files:
    result_files = glob.glob('**/results.csv', recursive=True)

print(f'Found result files: {result_files}')

for rf in result_files:
    print(f'\n--- {rf} ---')
    df = pd.read_csv(rf)
    print(df.to_string(index=False))

In [ ]:
# Also check history.json
import json
import glob

history_files = glob.glob('**/history.json', recursive=True)
for hf in history_files:
    print(f'\n--- {hf} ---')
    with open(hf) as f:
        history = json.load(f)
    for entry in history:
        print(entry)